`07_results_object_detection.ipynb` este notebook se encarga de parsear los resultados generados por los entrenamientos de EfficientDet y YOLO, consolidarlos en un DataFrame y exportar tablas formateadas para el documento final en LaTeX.

# Imports

In [36]:
import json
import pandas as pd
import os
from pathlib import Path

RUNS_DIR = Path('../runs')

# Funciones auxiliares

Con esta función se leera todos los summary_training.json de la carpeta de runs

In [37]:
def parse_experiment_results(root_dir):
    data_list = []
    
    json_files = list(root_dir.rglob('summary_training.json'))
    print(f"🔍 Encontrados {len(json_files)} archivos de resumen.")

    for json_path in json_files:
        try:
            with open(json_path, 'r') as f:
                data = json.load(f)
            

            parts = json_path.parts
            dataset_set = parts[3] if len(parts) > 3 else "N/A"
            experiment_name = parts[4] if len(parts) > 4 else "N/A"
            model_size = data.get("model_size_mb")
            if model_size is None:
                model_size = data.get("model_info_tl", {}).get("model_size_MB")

            total_params = data.get("model_info_tl", {}).get("total_params")
            trainable_tl = data.get("model_info_tl", {}).get("trainable_params")
            trainable_ft = data.get("model_info_ft", {}).get("trainable_params")
            test_map50 = data.get("test_mAP50")
            if test_map50 is None:
                test_map50 = data.get("results_test", {}).get("map_50")
            row = {
                "Model": data.get("model_name"),
                "Dataset_Set": dataset_set,
                "Experiment": experiment_name,
                "Img_Size": data.get("model_info_tl", {}).get("img_size"),
                "Total_Params": total_params,
                "Trainable_Params_TL": trainable_tl,
                "Trainable_Params_FT": trainable_ft,
                "Model_Size_MB": model_size,
                "Best_Val_mAP50_FT": data.get("best_val_map50_FT"),
                "Test_mAP50": test_map50,
                "Infer_Batch_Size": data.get("inference_stats", {}).get("batch_size"),
                "Mean_Batch_Time_Sec": data.get("inference_stats", {}).get("mean_batch_time_sec"),
                "Std_Batch_Time_Sec": data.get("inference_stats", {}).get("std_batch_time_sec"),
                "Mean_Img_Time_Sec": data.get("inference_stats", {}).get("mean_img_time_sec"),
                "FPS": data.get("inference_stats", {}).get("fps_batch_based")
            }
            
            data_list.append(row)
            
        except Exception as e:
            print(f"⚠️ Error parseando {json_path}: {e}")

    df = pd.DataFrame(data_list)
    return df

Leemos todos los summary_df

In [38]:
df_results = parse_experiment_results(RUNS_DIR)

# Ordenar para que sea más legible
if not df_results.empty:
    df_results = df_results.sort_values(by=['Model', 'Dataset_Set', 'Experiment']).reset_index(drop=True)

# Vista previa
df_results.head()

🔍 Encontrados 68 archivos de resumen.


,Model,Dataset_Set,Experiment,Img_Size,Total_Params,Trainable_Params_TL,Trainable_Params_FT,Model_Size_MB,Best_Val_mAP50_FT,Test_mAP50,Infer_Batch_Size,Mean_Batch_Time_Sec,Std_Batch_Time_Sec,Mean_Img_Time_Sec,FPS
0,tf_efficientdet_d0,set1_balanced_subsampled,augs_fold_1_config_without_mosaic,512.0,3829172,37503,3829172,NaN,0.404316,0.665662,16,0.028354,0.005148,0.001772,564.302009
1,tf_efficientdet_d0,set1_balanced_subsampled,fold_1_config,512.0,3829172,37503,3829172,15.537,0.565734,0.772863,16,0.028816,0.005291,0.001801,555.249756
2,tf_efficientdet_d0,set1_balanced_subsampled,fold_2_config,512.0,3829172,37503,3829172,15.537,0.759890,0.756412,16,0.028911,0.005373,0.001807,553.427515
3,tf_efficientdet_d0,set1_balanced_subsampled,fold_3_config,512.0,3829172,37503,3829172,15.537,0.762009,0.725770,16,0.028611,0.005275,0.001788,559.228877
4,tf_efficientdet_d0,set1_balanced_subsampled,fold_4_config,512.0,3829172,37503,3829172,15.537,0.711273,0.732936,16,0.029055,0.005374,0.001816,550.686885


# Experimento 1

En este experimento se mostrará los resultados de Efficientdet para el Fold 1, Escenario 1 sin augmentation y sin mosaicos, con augmentation y sin mosaicos y con augmentation y mosaicos.

In [39]:
filtro = (df_results['Model'] == 'tf_efficientdet_d2') & (df_results['Dataset_Set'] == 'set1_balanced_subsampled') & (df_results['Experiment'].str.contains('fold_1'))

df_filtrado = df_results[filtro]

df_exp1 = df_filtrado[['Experiment', 'Best_Val_mAP50_FT', 'Test_mAP50']]
df_exp1.to_latex(
            caption="exp1",
            label="tab:exp1"
)

latex_code = df_exp1.to_latex(
    caption="Comparación de resultados Experimento 1",
    label="tab:exp1",
    index=False
)
os.makedirs('docs/latex', exist_ok=True)

file_path = 'docs/latex/exp1.tex'
with open(file_path, 'w', encoding='utf-8') as f:
    f.write(latex_code)

print(f"✅ Archivo LaTeX guardado correctamente en: {file_path}")
display(df_exp1)

✅ Archivo LaTeX guardado correctamente en: docs/latex/exp1.tex


,Experiment,Best_Val_mAP50_FT,Test_mAP50
21,augs_fold_1_config_without_mosaic,0.637833,0.785975
22,fold_1_config,0.828922,0.851796
27,no_augs_fold_1_config,0.593099,0.742855


# Experimento 2

## Efficientdet d0

In [40]:
df_final = df_results[df_results['Model'].str.contains('tf_eff')].copy()

df_final = df_final[
    ~df_final['Experiment'].str.contains('without_mosaic|no_augs', case=False)
]

df_final['Model'] = df_final['Model'].str.replace('tf_efficientdet_', '').str.upper()
escenario_map = {
    'set1_balanced_subsampled': 'Escenario 1',
    'set2_balanced_full': 'Escenario 2',
    'set3_random_full': 'Escenario 3'
}
df_final['Dataset_Set'] = df_final['Dataset_Set'].map(escenario_map)
df_final['Experiment'] = df_final['Experiment'].str.replace('_config', '').str.replace('_', ' ').str.title()

# 2. Preparar columnas para la tabla
tabla_presentacion = df_final[[
    'Model', 'Dataset_Set', 'Experiment', 'Best_Val_mAP50_FT', 'Test_mAP50'
]].copy()
tabla_presentacion.columns = ['Modelo', 'Escenario', 'Fold', 'Best Val mAP@.5', 'Test mAP@.5']

# 3. EFECTO MULTIROW: Establecer índices jerárquicos y ordenar
# Esto agrupa visualmente las celdas repetidas de 'Modelo' y 'Escenario'
tabla_multirow = tabla_presentacion.set_index(['Modelo', 'Escenario']).sort_index()

tabla_final_visual = tabla_multirow.style.format({
    'Best Val mAP@.5': '{:.3f}',
    'Test mAP@.5': '{:.3f}'
})

# 5. Generar código LaTeX (Versión simplificada y segura)
# Quitamos multirow_heading_shape para evitar el TypeError
latex_code = tabla_final_visual.to_latex(
    caption="Comparación de resultados Experimento 2",
    label="tab:exp2",
    hrules=True,  # Necesita \usepackage{booktabs} en LaTeX
)

# 6. Guardar archivo
os.makedirs('docs/latex', exist_ok=True)
file_path = 'docs/latex/exp2_effdet.tex'

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(latex_code)

print(f"✅ Archivo LaTeX guardado en: {file_path}")
display(tabla_final_visual)

✅ Archivo LaTeX guardado en: docs/latex/exp2_effdet.tex


Resumen de efficientdet

In [41]:
df_best_folds = tabla_presentacion.sort_values('Best Val mAP@.5', ascending=False).groupby(['Modelo', 'Escenario']).head(1)
df_best_folds = df_best_folds.sort_values(['Modelo', 'Escenario'])
tabla_resumen_final = df_best_folds.set_index(['Modelo', 'Escenario'])

tabla_best_visual = tabla_resumen_final.style.format({
    'Best Val mAP@.5': '{:.4f}',
    'Test mAP@.5': '{:.4f}'
})

latex_best_code = tabla_best_visual.to_latex(
    caption="Resumen de mejores resultados por escenario (EfficientDet)",
    label="tab:best_results_effdet",
    hrules=True
)

os.makedirs('docs/latex', exist_ok=True)
file_path_best = 'docs/latex/exp2_best_folds_effdet.tex'

with open(file_path_best, 'w', encoding='utf-8') as f:
    f.write(latex_best_code)

print(f"✅ Tabla de mejores resultados guardada en: {file_path_best}")

display(tabla_best_visual)

✅ Tabla de mejores resultados guardada en: docs/latex/exp2_best_folds_effdet.tex


## YOLO26

In [43]:
df_final = df_results[df_results['Model'].str.contains('yolo26')].copy()

df_final = df_final[
    ~df_final['Experiment'].str.contains('without_mosaic|no_augs', case=False)
]

# df_final['Model'] = df_final['Model'].str.replace('yolo26', '').str.upper()
escenario_map = {
    'set1_balanced_subsampled': 'Escenario 1',
    'set2_balanced_full': 'Escenario 2',
    'set3_random_full': 'Escenario 3'
}
df_final['Dataset_Set'] = df_final['Dataset_Set'].map(escenario_map)
df_final['Experiment'] = df_final['Experiment'].str.replace('_config', '').str.replace('_', ' ').str.title()

tabla_presentacion = df_final[[
    'Model', 'Dataset_Set', 'Experiment', 'Best_Val_mAP50_FT', 'Test_mAP50'
]].copy()
tabla_presentacion.columns = ['Modelo', 'Escenario', 'Fold', 'Best Val mAP@.5', 'Test mAP@.5']


tabla_multirow = tabla_presentacion.set_index(['Modelo', 'Escenario']).sort_index()

tabla_final_visual = tabla_multirow.style.format({
    'Best Val mAP@.5': '{:.3f}',
    'Test mAP@.5': '{:.3f}'
})

latex_code = tabla_final_visual.to_latex(
    caption="Comparación de resultados Experimento 2",
    label="tab:exp2",
    hrules=True,
)

os.makedirs('docs/latex', exist_ok=True)
file_path = 'docs/latex/exp2_yolo26.tex'

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(latex_code)

print(f"✅ Archivo LaTeX guardado en: {file_path}")
display(tabla_final_visual)

✅ Archivo LaTeX guardado en: docs/latex/exp2_yolo26.tex


Resumen

In [44]:
df_best_folds = tabla_presentacion.sort_values('Best Val mAP@.5', ascending=False).groupby(['Modelo', 'Escenario']).head(1)
df_best_folds = df_best_folds.sort_values(['Modelo', 'Escenario'])
tabla_resumen_final = df_best_folds.set_index(['Modelo', 'Escenario'])

tabla_best_visual = tabla_resumen_final.style.format({
    'Best Val mAP@.5': '{:.4f}',
    'Test mAP@.5': '{:.4f}'
})

latex_best_code = tabla_best_visual.to_latex(
    caption="Resumen de mejores resultados por escenario (YOLO26)",
    label="tab:best_results_yolo26",
    hrules=True
)

os.makedirs('docs/latex', exist_ok=True)
file_path_best = 'docs/latex/exp2_best_folds_yolo.tex'

with open(file_path_best, 'w', encoding='utf-8') as f:
    f.write(latex_best_code)

print(f"✅ Tabla de mejores resultados guardada en: {file_path_best}")

display(tabla_best_visual)

✅ Tabla de mejores resultados guardada en: docs/latex/exp2_best_folds_yolo.tex


## Comparación entre los modelos

In [45]:
agregaciones = {
    'Total_Params': 'first',
    'Trainable_Params_TL': 'first',
    'Trainable_Params_FT': 'first',
    'Model_Size_MB': 'first',
    'Mean_Img_Time_Sec': 'mean'
}

df_tech_summary = df_results.groupby('Model').agg(agregaciones).reset_index()

df_tech_summary['Model'] = df_tech_summary['Model'].str.replace('tf_efficientdet_', '').str.upper()

df_tech_summary.columns = [
    'Arquitectura', 
    'Parámetros Totales', 
    'Parám. Entrenables (TL)', 
    'Parám. Entrenables (FT)', 
    'Tamaño (MB)', 
    'Latencia Imagen (s)', 
]

df_tech_visual = df_tech_summary.style.format({
    'Parámetros Totales': '{:,.0f}',
    'Parám. Entrenables (TL)': '{:,.0f}',
    'Parám. Entrenables (FT)': '{:,.0f}',
    'Tamaño (MB)': '{:.2f}',
    'Latencia Imagen (s)': '{:.4f}'
}).hide(axis='index')

latex_tech = df_tech_visual.to_latex(
    caption="Especificaciones técnicas y rendimiento de inferencia por arquitectura",
    label="tab:model_specs",
    hrules=True
)

file_path_tech = 'docs/latex/specs_modelos.tex'
with open(file_path_tech, 'w', encoding='utf-8') as f:
    f.write(latex_tech)

print(f"✅ Tabla técnica guardada en: {file_path_tech}")

display(df_tech_visual)

✅ Tabla técnica guardada en: docs/latex/specs_modelos.tex


Arquitectura,Parámetros Totales,Parám. Entrenables (TL),Parám. Entrenables (FT),Tamaño (MB),Latencia Imagen (s)
D0,"3,829,172","37,503","3,829,172",15.54,0.0019
D2,"8,008,560","97,839","8,008,560",32.55,0.0068
YOLO26M,"21,896,248","18,049,784","21,777,514",41.96,0.0045
YOLO26N,"2,572,280","1,901,208","2,504,970",5.11,0.0040
